In [1]:
from infostop import Infostop
import numpy as np
import polars as pl

import yaml

In [2]:
with open("05_detect_stops_config.yaml") as fp:
    config = yaml.safe_load(fp)


In [3]:
def initial_filter(df: pl.DataFrame, min_pings: int) -> pl.DataFrame:
    """Keep only devices with at least min_pings pings."""
    valid_devices = (
        df.group_by("device_id")
        .len()
        .filter(pl.col("len") >= min_pings)["device_id"]
        .to_list()
    )
    return df.filter(pl.col("device_id").is_in(valid_devices))


In [4]:
data = pl.read_parquet(
    "/mnt/common-hdd/group-dirs/tkom-2026-project/partitioned/00/pings.parquet"
)


In [5]:
filtered = initial_filter(data, min_pings=config["filtering"]["min_pings_per_day"])

In [6]:
filtered = filtered.with_columns(
    pl.col("timestamp").dt.epoch(time_unit="s").alias("epoch")
)

In [7]:
# stop detection process
params = config["infostop"]

model = Infostop(
    r1=params["r1"],
    r2=params["r2"],
    min_staying_time=params["min_staying_time"],
    max_time_between=params["max_time_between"],
    min_size=params["min_size"],
    distance_metric=params["distance_metric"],
)

In [8]:
# run infostop per device — data already sorted, no re-sort needed in loop
# catch devices with no stop events (all pings labelled -1)
results = []
for device_id, group in filtered.group_by("device_id"):
    coords = group[["latitude", "longitude", "epoch"]].to_numpy()
    try:
        labels = model.fit_predict(coords)
    except Exception:
        labels = np.full(len(coords), -1)
    group_ = group.with_columns(place_label=labels)

    results.append(group_)

clustered = pl.concat(results)
clustered = clustered.sort(["device_id", "timestamp"])

n_stopped = (clustered["place_label"] >= 0).sum()
print(
    f"Stop pings:    {n_stopped:,}  ({n_stopped / len(clustered):.1%} of filtered pings)"
)
print(f"Unique stops:  {clustered['place_label'].clip(lower_bound=0).n_unique():,}")
clustered.head(3)


Stop pings:    9,474,653  (61.3% of filtered pings)
Unique stops:  393


timestamp,device_id,device_aid_type,latitude,longitude,epoch,place_label
"datetime[μs, Europe/Budapest]",i64,str,f64,f64,i64,i64
2024-10-02 02:37:09 CEST,1,"""AAID""",47.47992,19.074155,1727829429,2
2024-10-02 03:17:49 CEST,1,"""AAID""",47.479995,19.074221,1727831869,2
2024-10-02 03:26:50 CEST,1,"""AAID""",47.476752,19.075154,1727832410,-1


In [9]:
clustered.drop("epoch").write_parquet(
    "/mnt/common-hdd/group-dirs/tkom-2026-project/clustered_pings/00.parquet"
)

In [ ]:
clustered = pl.read_parquet(
    "/mnt/common-hdd/group-dirs/tkom-2026-project/stops/00.parquet"
)

In [ ]:
clustered.filter(pl.col.device_id == 1)

timestamp,device_id,device_aid_type,latitude,longitude,stop_label
"datetime[μs, Europe/Budapest]",i64,str,f64,f64,i64
2024-10-02 02:37:09 CEST,1,"""AAID""",47.47992,19.074155,2
2024-10-02 03:17:49 CEST,1,"""AAID""",47.479995,19.074221,2
2024-10-02 03:26:50 CEST,1,"""AAID""",47.476752,19.075154,-1
2024-10-02 03:27:50 CEST,1,"""AAID""",47.47999,19.07423,-1
2024-10-02 04:06:48 CEST,1,"""AAID""",47.476752,19.075154,-1
…,…,…,…,…,…
2025-12-04 05:10:11 CET,1,"""AAID""",47.480054,19.074315,2
2025-12-04 07:22:49 CET,1,"""AAID""",47.479687,19.07482,2
2025-12-04 07:26:25 CET,1,"""AAID""",47.480067,19.074297,2


In [ ]:
clustered.group_by("place_label")

timestamp,device_id,device_aid_type,latitude,longitude,stop_label
"datetime[μs, Europe/Budapest]",i64,str,f64,f64,i64
2024-10-02 02:37:09 CEST,1,"""AAID""",47.47992,19.074155,2
2024-10-02 03:17:49 CEST,1,"""AAID""",47.479995,19.074221,2
2024-10-02 03:26:50 CEST,1,"""AAID""",47.476752,19.075154,-1
2024-10-02 03:27:50 CEST,1,"""AAID""",47.47999,19.07423,-1
2024-10-02 04:06:48 CEST,1,"""AAID""",47.476752,19.075154,-1
…,…,…,…,…,…
2024-11-18 09:08:35 CET,6584,"""AAID""",47.606608,19.270912,3
2024-11-18 09:13:46 CET,6584,"""AAID""",47.606841,19.27109,3
2024-11-18 09:33:03 CET,6584,"""AAID""",47.606869,19.270884,3


In [ ]:
import pandas as pd

In [ ]:
ref = pd.read_parquet(
    "/mnt/common-hdd/raw-sources/ubermedia/processed/initial_stops/complete/stop/00.parquet"
)

In [ ]:
ref.query("device_id=='00000b5ecb159b37f28ec4f68f6f7475487cc441'")

,center__lat,center__lon,device_id,first__lat,first__lon,interval__end,interval__start,is_between_stops,last__lat,last__lon,n_events,place_label,stop_number
0,47.454410,19.038114,00000b5ecb159b37f28ec4f68f6f7475487cc441,47.454411,19.038111,2019-06-16 22:39:37,2019-06-16 22:32:24,False,47.454410,19.038115,3,1,1
1,47.459316,19.040245,00000b5ecb159b37f28ec4f68f6f7475487cc441,47.454409,19.038088,2019-06-20 03:59:25,2019-06-17 02:32:48,True,47.478944,19.048871,5,-1,2
2,47.478935,19.048837,00000b5ecb159b37f28ec4f68f6f7475487cc441,47.478914,19.048882,2019-06-20 08:48:51,2019-06-20 08:34:47,False,47.478915,19.048873,21,3,3
3,47.475638,19.046743,00000b5ecb159b37f28ec4f68f6f7475487cc441,47.477749,19.047186,2019-06-20 13:20:21,2019-06-20 13:15:29,False,47.474130,19.046392,8,9,4
4,47.463446,19.042599,00000b5ecb159b37f28ec4f68f6f7475487cc441,47.467543,19.043971,2019-06-20 13:25:45,2019-06-20 13:21:54,True,47.459349,19.041227,8,-1,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...
129,47.504488,19.056887,00000b5ecb159b37f28ec4f68f6f7475487cc441,47.504488,19.056887,2019-08-26 19:05:18,2019-08-26 19:04:48,True,47.504488,19.056887,2,-1,130
130,47.478873,19.048878,00000b5ecb159b37f28ec4f68f6f7475487cc441,47.478932,19.048827,2019-09-24 15:35:51,2019-09-24 14:18:13,False,47.478969,19.048844,5,3,131
131,47.454451,19.038099,00000b5ecb159b37f28ec4f68f6f7475487cc441,47.454270,19.038172,2019-09-24 22:28:24,2019-09-24 22:26:30,True,47.454632,19.038025,4,-1,132
132,47.454440,19.038012,00000b5ecb159b37f28ec4f68f6f7475487cc441,47.454428,19.038064,2019-09-25 11:22:29,2019-09-25 09:06:55,False,47.454443,19.037962,7,1,133
